# ASR evaluation: Whisper large-v3 vs large-v3-turbo on our own recordings of team30

**What this notebook does.** Diagnostic, TODO §13 Phase 2. It measures the speech → English
stage on the team's 30 sentences, spoken by us:

1. **Record** each sentence once (interactive, 16 kHz mono). Existing files are skipped.
   **No microphone?** (a remote PC, 2026-09-24) §1b speaks the sentences with the Windows
   SAPI voices, clean and with added noise, and every later section scores each speech
   *source* separately. Synthetic speech is cleaner than a person reading, so its WER is an
   **optimistic bound**: it shows the pipeline works and how the models compare, not what a
   user will get.
2. **Transcribe** with every model in `asr.compare_models` (config): WER and latency.
3. **End to end:** gloss the ASR text vs gloss the true English. The gap is how much ASR
   errors cost downstream.

`large-v3` is the team's model; `large-v3-turbo` is what Workers AI hosts (§12.5).

| artifact | path |
|---|---|
| recordings | `data/raw/speech30/<id>.wav` |
| synthesized speech | `data/cache/synthesis/speech30_tts/<voice>/<clean\|snrNN>/<id>.wav` |
| transcripts | `data/cache/synthesis/asr/<model>.json` (your recordings) and `<model>__<source>.json` (synthesized), per sentence |
| WER / latency table | `data/cache/synthesis/asr/asr_scores.csv` |
| end-to-end gloss scores | `data/cache/synthesis/asr/e2e_scores.csv` |

**Resumable:** recording skips existing wavs, and transcription skips sentences already
in a model's JSON (written after each one).

**Design decisions vs the TODO spec:** Google Speech Commands is not used (keyword
spotting, no sentences; report §7.1). Your own recordings are the sentence-level speech
with transcripts that the plan called for. `best_of` is dropped (inert at temperature 0),
and peak normalization is applied to the audio Whisper actually gets (the team
notebook's was dead code).

## Setup

In [1]:
# ============================================================
# Imports, configs, paths
# ============================================================
import json
import os

import pandas as pd
import soundfile as sf

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR, ROOT_DIR
from sb.synthesize import asr, evalsets, metrics
from sb.synthesize.gloss import GlossEngine

CONFIG_DIR = EXPERIMENTS_DIR / "synthesis" / "configs"
SPEECH = json.loads((CONFIG_DIR / "speech.pipeline.json").read_text(encoding="utf-8"))
T2G = json.loads((CONFIG_DIR / "aslg.text2gloss.json").read_text(encoding="utf-8"))
REC_DIR = ROOT_DIR / SPEECH["recording"]["dir"]
OUT = CACHE_DIR / "synthesis" / "asr"
REC_DIR.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
T30 = evalsets.load("team30", T2G["eval"]["team30_version"])

MIC = asr.microphone()
print(f"microphone : {MIC or 'none (use §1b synthesized speech)'}")
print(f"recordings : {REC_DIR} ({len(list(REC_DIR.glob('*.wav')))}/{len(T30)} present)")
print(f"results    : {OUT}")
print(f"models     : {SPEECH['asr']['compare_models']}")

microphone : none (use §1b synthesized speech)
recordings : C:\Users\user2\repository\data\raw\speech30 (0/30 present)
results    : C:\Users\user2\repository\data\cache\synthesis\asr
models     : ['large-v3', 'large-v3-turbo']


## 1. Record the 30 sentences

Set `RECORD = True`, run the cell, and read each sentence when it prints. Each recording
lasts `SECONDS`. Re-run to fill gaps; `REDO` lists ids to re-record.

In [2]:
# ============================================================
# Interactive recorder (skips existing wavs)
# ============================================================
RECORD = False   # True -> record every missing sentence
SECONDS = SPEECH["recording"]["seconds"]
REDO = []        # e.g. [7, 19] -> record these again

if RECORD and MIC is None:
    print("no microphone on this machine -- skipped; §1b synthesizes the sentences instead")
elif RECORD:
    import time

    for _, r in T30.iterrows():
        path = REC_DIR / f"{r.id:02d}.wav"
        if path.exists() and r.id not in REDO:
            continue
        print(f"\n[{r.id}/30] in 2 s, read aloud:  {r.english}")
        time.sleep(2)
        audio = asr.record(SECONDS)
        tmp = path.with_suffix(".tmp.wav")
        sf.write(tmp, audio, asr.SAMPLE_RATE)
        os.replace(tmp, path)
        print(f"   saved {path.name} (peak {abs(audio).max():.2f})")
print(f"{len(list(REC_DIR.glob('??.wav')))}/{len(T30)} recordings present")

0/30 recordings present


## 1b. No microphone: synthesize the 30 sentences

Every enabled Windows SAPI voice reads every sentence (`sb.synthesize.tts`). Each clean file
also gets noisy copies at the configured SNRs (`tts_fallback.snr_db`, seeded per voice/SNR/id).
Existing files are skipped. Each `<voice>/<clean|snrNN>` folder is one **source** below.

In [3]:
# ============================================================
# SAPI voices x (clean + noise levels) -> data/cache/synthesis/speech30_tts/
# ============================================================
import re

import numpy as np
from tqdm.auto import tqdm

from sb.synthesize import tts

TTS = SPEECH["tts_fallback"]
TTS_DIR = ROOT_DIR / TTS["dir"]
VOICES = tts.sapi_voices() if TTS["voices"] == "all" else TTS["voices"]


def slug(v: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", v.lower()).strip("-")


def level(snr) -> str:
    return "clean" if snr is None else f"snr{int(snr)}"


jobs = [(v, snr, r) for v in VOICES for snr in TTS["snr_db"] for _, r in T30.iterrows()
        if not (TTS_DIR / slug(v) / level(snr) / f"{r.id:02d}.wav").exists()]
for v, snr, r in tqdm(jobs, desc="synthesize"):
    clean = TTS_DIR / slug(v) / "clean" / f"{r.id:02d}.wav"
    if not clean.exists():
        tts.sapi_speak(r.english, clean, v, TTS["rate"])
    if snr is not None:
        audio, sr = sf.read(clean, dtype="float32")
        rng = np.random.default_rng([TTS["seed"], VOICES.index(v), int(snr), int(r.id)])
        out = TTS_DIR / slug(v) / level(snr) / f"{r.id:02d}.wav"
        out.parent.mkdir(parents=True, exist_ok=True)
        tmp = out.with_suffix(".tmp.wav")
        sf.write(tmp, tts.add_noise(audio, snr, rng), sr)
        os.replace(tmp, out)

SOURCES = {"human": REC_DIR} | {f"tts-{slug(v)}-{level(s)}": TTS_DIR / slug(v) / level(s)
                                for v in VOICES for s in TTS["snr_db"]}
print({k: len(list(d.glob("??.wav"))) for k, d in SOURCES.items()})

synthesize: 0it [00:00, ?it/s]

{'human': 0, 'tts-microsoft-david-desktop-clean': 30, 'tts-microsoft-david-desktop-snr20': 30, 'tts-microsoft-david-desktop-snr10': 30, 'tts-microsoft-david-desktop-snr5': 30, 'tts-microsoft-david-desktop-snr0': 30, 'tts-microsoft-zira-desktop-clean': 30, 'tts-microsoft-zira-desktop-snr20': 30, 'tts-microsoft-zira-desktop-snr10': 30, 'tts-microsoft-zira-desktop-snr5': 30, 'tts-microsoft-zira-desktop-snr0': 30}


## 2. Transcribe with each model

Each model's per-sentence transcripts go to `<model>.json` as they are made. WER uses the
team's normalization (lowercase, no punctuation). Numbers are compared as written, so
"3 o'clock" vs "three o'clock" counts as an error: that is a real downstream difference,
since the gloss engines see the digit.

In [4]:
# ============================================================
# Transcribe every source (resumable per model x source) + WER / latency
# ============================================================
from tqdm.auto import tqdm


def transcript_path(model: str, source: str):
    return OUT / (f"{model}.json" if source == "human" else f"{model}__{source}.json")


rows = []
for model in SPEECH["asr"]["compare_models"]:
    tr = None
    for source, d in SOURCES.items():
        wavs = {int(p.stem): p for p in sorted(d.glob("??.wav"))}
        if not wavs:
            continue
        path = transcript_path(model, source)
        done = json.loads(path.read_text()) if path.exists() else {}
        todo = [i for i in wavs if str(i) not in done]
        if todo:
            tr = tr or asr.Transcriber.from_config(SPEECH["asr"], model=model)
            for i in tqdm(todo, desc=f"ASR {model} {source}"):
                t = tr(wavs[i])
                done[str(i)] = {"text": t.text, "seconds": t.seconds, "audio_seconds": t.audio_seconds}
                tmp = path.with_suffix(".tmp")
                tmp.write_text(json.dumps(done, indent=1))
                os.replace(tmp, path)
        ids = sorted(int(k) for k in done)
        refs = [T30.english[i - 1] for i in ids]
        hyps = [done[str(i)]["text"] for i in ids]
        rows.append({"source": source, "model": model, "n": len(ids), "wer": metrics.asr_wer(refs, hyps),
                     "sent_exact": sum(metrics.normalize_text(a) == metrics.normalize_text(b) for a, b in zip(refs, hyps)) / max(len(ids), 1),
                     "latency_median_s": pd.Series([done[str(i)]["seconds"] for i in ids]).median()})
    del tr
tab = pd.DataFrame(rows).set_index(["source", "model"])
tab.to_csv(OUT / "asr_scores.csv")
tab.round(3)

,,n,wer,sent_exact,latency_median_s
source,model,,,,
tts-microsoft-david-desktop-clean,large-v3,30,0.010,0.933,0.211
tts-microsoft-david-desktop-snr20,large-v3,30,0.010,0.933,0.204
tts-microsoft-david-desktop-snr10,large-v3,30,0.010,0.933,0.201
tts-microsoft-david-desktop-snr5,large-v3,30,0.010,0.933,0.198
tts-microsoft-david-desktop-snr0,large-v3,30,0.026,0.833,0.200
tts-microsoft-zira-desktop-clean,large-v3,30,0.010,0.933,0.205
tts-microsoft-zira-desktop-snr20,large-v3,30,0.010,0.933,0.204
tts-microsoft-zira-desktop-snr10,large-v3,30,0.010,0.933,0.204
tts-microsoft-zira-desktop-snr5,large-v3,30,0.015,0.900,0.190


## 3. End to end: does ASR error survive into the gloss?

For every engine, gloss (a) the true English and (b) each model's transcript, and score
both against the draft references. The difference is the ASR cost. It is often smaller
than the WER, because many ASR errors are in words the gloss drops anyway (articles,
auxiliaries).

The pipeline spells out digits before glossing (`asr.spell_numbers`, 2026-09-24): Whisper writes `3 o'clock`, but the engines expect `THREE`. `asr_cost_digits_unspelled` is the cost without that fix.

In [5]:
# ============================================================
# Gloss of ASR text vs gloss of true text, per source
# ============================================================
refs_all = {i + 1: metrics.normalize_gloss(g) for i, g in enumerate(T30.reference_draft)}
engines = {name: GlossEngine(name, T2G["t5"]) for name in T2G["eval"]["engines"]}
true_gloss = {name: {i + 1: metrics.normalize_gloss(r["gloss"])
                     for i, r in enumerate(eng.gloss_sentences(list(T30.english)))} for name, eng in engines.items()}
rows = []
for model in SPEECH["asr"]["compare_models"]:
    for source in SOURCES:
        path = transcript_path(model, source)
        if not path.exists():
            continue
        done = json.loads(path.read_text())
        ids = sorted(int(k) for k in done)
        for name, eng in engines.items():
            refs = [refs_all[i] for i in ids]
            true = [true_gloss[name][i] for i in ids]
            raw = [metrics.normalize_gloss(eng(done[str(i)]["text"]).gloss) for i in ids]
            heard = [metrics.normalize_gloss(eng(asr.spell_numbers(done[str(i)]["text"])).gloss) for i in ids]
            a, b = metrics.corpus_scores(refs, true, meteor=False), metrics.corpus_scores(refs, heard, meteor=False)
            r = metrics.corpus_scores(refs, raw, meteor=False)
            rows.append({"source": source, "asr": model, "engine": name, "gloss_wer_true_text": a["gloss_wer"],
                         "gloss_wer_asr_text": b["gloss_wer"], "asr_cost": b["gloss_wer"] - a["gloss_wer"],
                         "asr_cost_digits_unspelled": r["gloss_wer"] - a["gloss_wer"],
                         "bleu4_true_text": a["bleu4"], "bleu4_asr_text": b["bleu4"]})
e2e = pd.DataFrame(rows)
e2e.to_csv(OUT / "e2e_scores.csv", index=False)
(e2e.pivot_table(index=["source", "asr"], columns="engine", values="asr_cost").round(3)
 if len(e2e) else "no transcripts yet -- run §1/§1b and §2")

engine                                            hybrid_guarded  hybrid_team  \
source                            asr                                           
tts-microsoft-david-desktop-clean large-v3                 0.000        0.000   
                                  large-v3-turbo           0.000        0.000   
tts-microsoft-david-desktop-snr0  large-v3                 0.013        0.013   
                                  large-v3-turbo           0.013        0.013   
tts-microsoft-david-desktop-snr10 large-v3                 0.000        0.000   
                                  large-v3-turbo           0.000        0.000   
tts-microsoft-david-desktop-snr20 large-v3                 0.000        0.000   
                                  large-v3-turbo           0.000        0.000   
tts-microsoft-david-desktop-snr5  large-v3                 0.000        0.000   
                                  large-v3-turbo           0.000        0.000   
tts-microsoft-zira-desktop-clean  large-v3                 0.000        0.000   
                                  large-v3-turbo           0.000        0.000   
tts-microsoft-zira-desktop-snr0   large-v3                 0.020        0.020   
                                  large-v3-turbo           0.026        0.026   
tts-microsoft-zira-desktop-snr10  large-v3                 0.000        0.000   
                                  large-v3-turbo           0.007        0.007   
tts-microsoft-zira-desktop-snr20  large-v3                 0.000        0.000   
                                  large-v3-turbo           0.000        0.000   
tts-microsoft-zira-desktop-snr5   large-v3                 0.000        0.000   
                                  large-v3-turbo           0.013        0.013   

engine                                            rules_v1  rules_v2  
source                            asr                                 
tts-microsoft-david-desktop-clean large-v3           0.000     0.000  
                                  large-v3-turbo     0.000     0.000  
tts-microsoft-david-desktop-snr0  large-v3           0.013     0.013  
                                  large-v3-turbo     0.013     0.013  
tts-microsoft-david-desktop-snr10 large-v3           0.000     0.000  
                                  large-v3-turbo     0.000     0.000  
tts-microsoft-david-desktop-snr20 large-v3           0.000     0.000  
                                  large-v3-turbo     0.000     0.000  
tts-microsoft-david-desktop-snr5  large-v3           0.000     0.000  
                                  large-v3-turbo     0.000     0.000  
tts-microsoft-zira-desktop-clean  large-v3           0.000     0.000  
                                  large-v3-turbo     0.000     0.000  
tts-microsoft-zira-desktop-snr0   large-v3           0.020     0.020  
                                  large-v3-turbo     0.026     0.026  
tts-microsoft-zira-desktop-snr10  large-v3           0.000     0.000  
                                  large-v3-turbo     0.007     0.007  
tts-microsoft-zira-desktop-snr20  large-v3           0.000     0.000  
                                  large-v3-turbo     0.000     0.000  
tts-microsoft-zira-desktop-snr5   large-v3           0.000     0.000  
                                  large-v3-turbo     0.013     0.013